In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/data.yaml
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/16_08_47_329.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/09_44_12_239.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/12_40_30_453.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/11_35_22_749.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/17_59_51_046.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/14_40_18_030.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/17_28_15_343.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/15_37_03_671.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/17_11_47_187.txt
/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO/labels/val/07_16_10_859.txt
/kaggle/input/dat

In [2]:
print("Notebook is working!")

Notebook is working!


In [3]:
!pip install -q -U ultralytics pycocotools pandas matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.11.2 which is incompatible.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you ha

In [4]:
import torch
import ultralytics
import pandas as pd
import matplotlib
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.178
PyTorch: 2.11.0+cu128
Pandas: 2.3.3
Matplotlib: 3.10.0
CUDA available: True
GPU count: 2
GPU: Tesla T4


In [5]:
from pathlib import Path
import yaml

DATASET_ROOT = Path(
    "/kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO"
)
DATA_YAML = Path("/kaggle/working/CLCXray_YOLO_data.yaml")

assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"

data = {
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 12,
    "names": [
        "blade", "scissors", "knife", "dagger",
        "SwissArmyKnife", "PlasticBottle", "Cans",
        "VacuumCup", "GlassBottle", "CartonDrinks",
        "Tin", "SprayCans"
    ]
}

for split in ["train", "val", "test"]:
    folder = DATASET_ROOT / "images" / split
    assert folder.exists(), f"Missing {split} folder"
    print(f"{split}: {len(list(folder.iterdir()))} files")

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("\nDataset YAML created:", DATA_YAML)

train: 3996 files
val: 956 files
test: 956 files

Dataset YAML created: /kaggle/working/CLCXray_YOLO_data.yaml


In [6]:
from pathlib import Path
import shutil
import zipfile
import time

RUN_DIR = Path("/kaggle/working/CLCXray_runs/YOLO26n_scratch")
BACKUP_DIR = Path("/kaggle/working/YOLO26n_backups")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def backup_yolo_results():
    if not RUN_DIR.exists():
        print("Training output not created yet.")
        return None

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    zip_path = BACKUP_DIR / f"YOLO26n_backup_{timestamp}.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for file in RUN_DIR.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(RUN_DIR.parent))

    print("Backup created:", zip_path)
    print("Size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))
    return zip_path

print("Backup folder ready:", BACKUP_DIR)
print("Expected training folder:", RUN_DIR)

Backup folder ready: /kaggle/working/YOLO26n_backups
Expected training folder: /kaggle/working/CLCXray_runs/YOLO26n_scratch


In [7]:
from ultralytics import YOLO
from pathlib import Path
import zipfile

RUN_DIR = Path("/kaggle/working/CLCXray_runs/YOLO26n_scratch")
BACKUP_DIR = Path("/kaggle/working/YOLO26n_backups")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def save_periodic_backup(trainer):
    epoch = trainer.epoch + 1

    if epoch % 10 != 0:
        return

    if not RUN_DIR.exists():
        return

    zip_path = BACKUP_DIR / f"YOLO26n_epoch_{epoch:03d}.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for file in RUN_DIR.rglob("*"):
            if file.is_file():
                z.write(file, file.relative_to(RUN_DIR.parent))

    print(f"Backup created at epoch {epoch}: {zip_path}")

model = YOLO("yolo26n.yaml")
model.add_callback("on_fit_epoch_end", save_periodic_backup)

In [8]:
from ultralytics import YOLO
from pathlib import Path
import torch
import shutil
import time

assert torch.cuda.is_available(), "GPU unavailable!"
assert DATA_YAML.exists(), "Dataset YAML missing!"

# Build architecture from YAML; do not load pretrained weights
model = YOLO("yolo26n.yaml")

results = model.train(
    data=str(DATA_YAML),
    epochs=200,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    pretrained=False,
    seed=42,
    deterministic=True,
    patience=25,
    save=True,
    save_period=10,
    plots=True,
    amp=True,
    fliplr=0.5,
    flipud=0.0,
    degrees=10.0,
    translate=0.05,
    scale=0.10,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    project="/kaggle/working/CLCXray_runs",
    name="YOLO26n_scratch",
    exist_ok=True,
)

print("Training finished!")
print("Results saved to:", results.save_dir)

Ultralytics 8.4.178 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/CLCXray_YOLO_data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.yaml, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=YOLO26n_scratch, nbs=64, nms=None, opset=None, 

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/200      3.88G      2.686      7.517    0.04132         29        640: 100% ━━━━━━━━━━━━ 250/250 5.2it/s 47.8s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 4.1it/s 7.3s
                   all        956       2317      0.313      0.236      0.103     0.0266

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/200      3.88G      1.997      3.515    0.02838         30        640: 100% ━━━━━━━━━━━━ 250/250 5.4it/s 45.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 4.1it/s 7.3s
                   all        956       2317      0.561       0.32      0.293      0.114

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/200      3.88G      1.786      2.238    0.02457         19        640: 100% ━━━━━━━━━━━━ 250/250 5.4it/s 46.5s
                 Class     Images  Instances      Box

In [9]:
import shutil
from pathlib import Path

run_dir = Path("/kaggle/working/CLCXray_runs/YOLO26n_scratch")
zip_path = "/kaggle/working/YOLO26n_scratch_results"

shutil.make_archive(
    zip_path,
    "zip",
    root_dir=run_dir
)

print("Backup created:", zip_path + ".zip")

Backup created: /kaggle/working/YOLO26n_scratch_results.zip
